# Personal AI Assistant for Students

Plan study sessions around deadlines and available time. A supervisor delegates to specialists in study planning, the timetable, and course materials. Python calculates the schedule and the supervisor explains the results.

## Getting started

1. Open PowerShell in `D:\Echo\ai_agent` and run `py -m venv .venv`.
2. Run `.\.venv\Scripts\python.exe -m pip install -r requirement.txt`.
3. Run `.\.venv\Scripts\python.exe -m jupyterlab study_planner.ipynb`.
4. Put your Groq API key in `.env` as `GROQ_API_KEY=...`, then run the cells in order. [Create a Groq API key](https://console.groq.com/keys)

Run the cells in order. Availability and deadlines start as examples in `sample_week.json`. Tell the assistant your actual study windows and it will save them. All times use Africa/Cairo.

## Data and scheduling

The tools read assignments and availability from `sample_week.json`. You can edit that file or tell the assistant your study windows. The assistant removes scheduled classes when it builds a study plan.

In [11]:
import json
from datetime import date, datetime, timedelta
from pathlib import Path


def parse_date(value: str) -> date:
    try:
        day = date.fromisoformat(value)
        if day.isoformat() == value:
            return day
    except (TypeError, ValueError):
        pass
    raise ValueError("Use a valid date in YYYY-MM-DD format.")


def format_time(value: datetime) -> str:
    return value.isoformat(timespec="minutes")


def parse_time(value: str) -> datetime:
    try:
        instant = datetime.fromisoformat(value)
        if instant.tzinfo is None and format_time(instant) == value:
            return instant
    except (TypeError, ValueError):
        pass
    raise ValueError("Use a valid Cairo time in YYYY-MM-DDTHH:MM format.")


def load_sample(path: Path) -> dict:
    """Read and validate the assignments and free windows."""
    try:
        data = json.loads(path.read_text(encoding="utf-8"))
        assignments = data["assignments"]
        windows = data["free_slots"]
        if not isinstance(assignments, list) or not isinstance(windows, list):
            raise ValueError("assignments and free_slots must be arrays.")

        seen_ids = set()
        for task in assignments:
            task_id, title = task["id"], task["task"]
            if not isinstance(task_id, str) or not task_id.strip() or task_id in seen_ids:
                raise ValueError("Every assignment needs a unique, non-empty id.")
            if not isinstance(title, str) or not title.strip():
                raise ValueError(f"Assignment {task_id!r} needs a task title.")
            parse_time(task["due"])
            minutes = task["estimated_minutes"]
            if type(minutes) is not int or minutes < 1:
                raise ValueError(f"Assignment {task_id!r} needs positive whole estimated_minutes.")
            seen_ids.add(task_id)

        for window in windows:
            start, end = parse_time(window["start"]), parse_time(window["end"])
            if start >= end or start.date() != end.date():
                raise ValueError("Free windows must start and end on the same day, in time order.")
    except (OSError, KeyError, TypeError, ValueError) as error:
        raise ValueError(f"Check {path.name}: {error}") from error
    return data


sample = load_sample(Path("sample_week.json"))

In [12]:
def validate_range(start_date: str, end_date: str) -> None:
    if parse_date(start_date) > parse_date(end_date):
        raise ValueError("start_date must fall on or before end_date.")


def get_deadlines(start_date: str, end_date: str) -> list[dict]:
    """List deadlines and estimated work between two inclusive YYYY-MM-DD dates."""
    validate_range(start_date, end_date)
    tasks = [
        {
            "id": task["id"],
            "task": task["task"],
            "due": task["due"],
            "estimated_minutes": task["estimated_minutes"],
        }
        for task in sample["assignments"]
        if start_date <= task["due"][:10] <= end_date
    ]
    return sorted(tasks, key=lambda task: (task["due"], task["id"]))


def get_free_slots(start_date: str, end_date: str) -> list[dict]:
    """List merged study windows between two inclusive YYYY-MM-DD dates."""
    validate_range(start_date, end_date)
    windows = sorted(sample["free_slots"], key=lambda window: window["start"])
    merged = []
    for window in windows:
        start, end = window["start"], window["end"]
        if not start_date <= start[:10] <= end_date:
            continue
        if merged and start <= merged[-1]["end"]:
            merged[-1]["end"] = max(merged[-1]["end"], end)
        else:
            merged.append({"start": start, "end": end})
    return merged


def build_study_plan(start_date: str, end_date: str, daily_limit_minutes: int = 120) -> dict:
    """Schedule study for deadlines in the inclusive YYYY-MM-DD range and report unfinished work."""
    if type(daily_limit_minutes) is not int or daily_limit_minutes < 1:
        raise ValueError("daily_limit_minutes must be a positive whole number.")

    windows = available_windows(start_date, end_date)
    tasks = get_deadlines(start_date, end_date)
    due_times = {task["id"]: parse_time(task["due"]) for task in tasks}
    remaining = {task["id"]: task["estimated_minutes"] for task in tasks}
    used_by_day = {}
    blocks = []

    for window in windows:
        cursor, slot_end = parse_time(window["start"]), parse_time(window["end"])
        day = cursor.date()
        used = used_by_day.get(day, 0)
        for task in tasks:
            if cursor >= slot_end or used >= daily_limit_minutes:
                break
            task_id = task["id"]
            cutoff = min(slot_end, due_times[task_id])
            available = (cutoff - cursor) // timedelta(minutes=1)
            minutes = min(remaining[task_id], available, daily_limit_minutes - used)
            if minutes <= 0:
                continue

            end = cursor + timedelta(minutes=minutes)
            blocks.append({
                "task_id": task_id,
                "task": task["task"],
                "start": format_time(cursor),
                "end": format_time(end),
                "minutes": minutes,
            })
            remaining[task_id] -= minutes
            cursor = end
            used += minutes
        used_by_day[day] = used

    unfinished = [
        {"task_id": task["id"], "task": task["task"], "remaining_minutes": remaining[task["id"]]}
        for task in tasks
        if remaining[task["id"]]
    ]
    return {"blocks": blocks, "unfinished": unfinished}

def set_study_availability(day: str, windows: list[dict[str, str]]) -> dict:
    """Replace a date's study windows. Each window has start and end in HH:MM, Cairo time. An empty list clears the date."""
    import os
    import tempfile

    parse_date(day)
    updated = []
    for window in windows:
        start = parse_time(f"{day}T{window['start']}")
        end = parse_time(f"{day}T{window['end']}")
        if start >= end:
            raise ValueError("Each window must end after it starts on the same day.")
        updated.append({"start": format_time(start), "end": format_time(end)})

    path = Path("sample_week.json")
    data = load_sample(path)
    data["free_slots"] = [slot for slot in data["free_slots"] if slot["start"][:10] != day] + updated
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=path.parent, delete=False) as file:
            temporary = Path(file.name)
            json.dump(data, file, ensure_ascii=False, indent=2)
            file.write("\n")
        os.replace(temporary, path)
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)
    sample.clear()
    sample.update(data)
    return {"day": day, "windows": updated, "message": "Availability saved."}


## Timetable and course materials

The next cell loads your weekly timetable and builds a local index of the files in `course_materials/`. Rerun it after adding files.

In [20]:
from llama_index.core import SimpleKeywordTableIndex
from llama_index.core.llms import MockLLM
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import Document
from pypdf import PdfReader

timetable_path = Path("data/timetable.json")
timetable = json.loads(timetable_path.read_text(encoding="utf-8")) if timetable_path.exists() else []

def search_timetable(day: str = None, course: str = None, session_type: str = None) -> list[dict]:
    """Find weekly university sessions by weekday, course, or session type; returns timetable row references."""
    filters = {"day": day, "course": course, "session_type": session_type}
    return [row for row in timetable if all(value is None or value.casefold() in row[key].casefold() for key, value in filters.items())]

def available_windows(start_date: str, end_date: str) -> list[dict]:
    """Subtract weekly university classes from the student's free windows."""
    result = []
    for window in get_free_slots(start_date, end_date):
        start, end = parse_time(window["start"]), parse_time(window["end"])
        spans = [(start, end)]
        for row in search_timetable(day=start.strftime("%A")):
            class_start = datetime.fromisoformat(f"{start.date()}T{row['start']}")
            class_end = datetime.fromisoformat(f"{start.date()}T{row['end']}")
            remaining = []
            for left, right in spans:
                if class_end <= left or class_start >= right:
                    remaining.append((left, right))
                else:
                    if left < class_start:
                        remaining.append((left, class_start))
                    if class_end < right:
                        remaining.append((class_end, right))
            spans = remaining
        result.extend({"start": format_time(left), "end": format_time(right)} for left, right in spans)
    return result

def build_course_index():
    """Read course files and build a local keyword index. Rerun after changing files."""
    documents = []
    warnings = []
    for path in sorted(Path("course_materials").rglob("*")):
        if not path.is_file() or path.suffix.lower() not in {".pdf", ".txt", ".md"}:
            continue
        try:
            if path.suffix.lower() == ".pdf":
                pages = [(f"page {i}", page.extract_text() or "") for i, page in enumerate(PdfReader(path).pages, 1)]
            else:
                pages = [("text", path.read_text(encoding="utf-8"))]
            if not any(text.strip() for _, text in pages):
                warnings.append(f"{path.name}: no selectable text; provide a text version.")
            for section, text in pages:
                if text.strip():
                    documents.append(Document(text=text, metadata={"source": str(path), "section": section}))
        except Exception as error:
            warnings.append(f"{path.name}: could not read file ({error}).")
    nodes = SentenceSplitter(chunk_size=512, chunk_overlap=50).get_nodes_from_documents(documents)
    index = SimpleKeywordTableIndex(nodes, llm=MockLLM(), max_keywords_per_chunk=512) if nodes else None
    return index, warnings

course_index, course_warnings = build_course_index()

def search_course_materials(query: str) -> dict:
    """Find relevant course excerpts and source references for a student's question."""
    if not query.strip():
        return {"excerpts": [], "message": "Provide a course question."}
    if course_index is None:
        return {"excerpts": [], "message": "No readable course materials. Add selectable-text PDFs, Markdown, or text files and rebuild the index.", "warnings": course_warnings}
    nodes = course_index.as_retriever(num_chunks_per_query=4, max_keywords_per_query=50).retrieve(query)
    excerpts = [{"source": node.metadata["source"], "section": node.metadata["section"], "text": node.node.text} for node in nodes]
    return {"excerpts": excerpts, "message": "Use only relevant evidence. If it does not answer the question, say so.", "warnings": course_warnings}

Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 30 0 (offset 0)
Ignoring wrong pointing object 33 0 (offset 0)
Ignoring wrong pointing object 75 0 (offset 0)
Ignoring wrong pointing object 97 0 (offset 0)
Ignoring wrong pointing object 130 0 (offset 0)
Ignoring wrong pointing object 132 0 (offset 0)
Ignoring wrong pointing object 134 0 (offset 0)
Ignoring wrong pointing object 166 0 (offset 0)
Ignoring wrong pointing object 171 0 (offset 0)
Ignoring wrong pointing object 192 0 (offset 0)
Ignoring wrong pointing object 227 0 (offset 0)
Ignoring wrong pointing object 229 0 (offset 0)
Ignoring wrong pointing object 281 0 (offset 0)
Ignoring wrong pointing object 292 0 (offset 0)
Ignoring wrong pointing object 294 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 30 0 (offset 0)
Ignoring wrong pointing object 35 0 (offset 0)
Ignoring wrong pointing object 59 0 (offset 0)
Ignoring wrong pointing object 70 0 (offset 0)
Ig

## Supervisor and specialists

`student_supervisor.py` defines the prompts and delegation tools. The supervisor keeps the conversation history. Each specialist receives a self-contained subtask and uses its own tools. The trace below shows which agent called each tool.

In [ ]:
from student_supervisor import create_supervisor, run_agent

## Check the example

This check uses the example data. Run it before changing the example windows, or skip it when using your own data.

In [ ]:
start_date, end_date = "2026-11-23", "2026-11-29"
assert get_free_slots(start_date, start_date) == [
    {"start": "2026-11-23T18:00", "end": "2026-11-23T20:00"}
]

sample_plan = build_study_plan(start_date, end_date, daily_limit_minutes=90)
assert [block["minutes"] for block in sample_plan["blocks"]] == [90, 30, 30]
assert sample_plan["blocks"][-1]["end"] == "2026-11-24T19:00"
assert sample_plan["unfinished"] == [
    {"task_id": "db-report", "task": "Database report", "remaining_minutes": 120}
]

try:
    build_study_plan("2026-11-31", end_date)
except ValueError:
    pass
else:
    raise AssertionError("An invalid calendar date was accepted.")

print(json.dumps(sample_plan, indent=2))

## Talk to the assistant

Run the setup cell once, then edit and rerun the request cell to continue the conversation. The example uses both course materials and study planning. To replace saved availability, specify the date and every window to keep.

In [ ]:
from dotenv import dotenv_values
from IPython.display import Markdown, display
from llama_index.llms.groq import Groq


def show_tool_result(agent_name, event):
    print(f"[{agent_name}] {event.tool_name}({event.tool_kwargs})")
    print(event.tool_output.content)


def create_agent():
    api_key = (dotenv_values(".env").get("GROQ_API_KEY") or "").strip()
    if not api_key:
        raise ValueError("Add GROQ_API_KEY to the .env file, then rerun this cell.")
    return create_supervisor(
        Groq(model="openai/gpt-oss-20b", api_key=api_key),
        get_deadlines=get_deadlines,
        get_free_slots=get_free_slots,
        set_study_availability=set_study_availability,
        build_study_plan=build_study_plan,
        search_timetable=search_timetable,
        search_course_materials=search_course_materials,
        trace=show_tool_result,
    )


agent = create_agent()
conversation = []


async def ask_planner(question: str) -> None:
    response = await run_agent(agent, question, history=conversation, trace=show_tool_result)
    display(Markdown(response))

In [ ]:
request = "Explain process states using my OS lectures, then plan my assignment work for 2026-11-24 using my saved availability."
await ask_planner(request)